# SMIRK – 3D Facial Expression Reconstruction (CVPR 2024)

**Paper:** [SMIRK: 3D Facial Expressions Through Analysis-by-Neural-Synthesis](https://arxiv.org/abs/2404.04104)  
**Repo:** https://github.com/georgeretsi/smirk  

This notebook runs **image inference** using the pretrained SMIRK model — no training required.

### What you need before running
1. **GPU runtime** — Runtime → Change runtime type → T4 GPU (free tier works fine)
2. **FLAME 2020 model** — Download `FLAME2020.zip` (free) from https://flame.is.tue.mpg.de → upload it when prompted in Cell 5.

All other dependencies (PyTorch3D, mediapipe, SMIRK checkpoint) are downloaded automatically.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to Runtime → Change runtime type → Hardware accelerator → GPU (T4)."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
import sys, subprocess, urllib.request, os, importlib

def run(cmd, env=None):
    e = {**os.environ, **(env or {})}
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=e)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

def is_installed(module_name):
    return importlib.util.find_spec(module_name) is not None

import torch
tv = torch.__version__.split('+')
torch_ver = tv[0].replace('.', '')
cu_tag = f"cu{torch.version.cuda.replace('.', '')}"
py_ver  = f"py{sys.version_info.major}{sys.version_info.minor}"
wheel_tag = f"{py_ver}_{cu_tag}_pyt{torch_ver}"
wheel_url = f"https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/{wheel_tag}/download.html"

print(f"Detected: torch={torch.__version__}, {py_ver}, {cu_tag}")

# ── numpy ──────────────────────────────────────────────────────────────────────
# Must be installed before everything else — FLAME pickle requires numpy<2.0
# at the C level (ndarray subclass reconstruction fails with numpy 2.x in memory)
print("Pinning numpy<2.0...")
run("pip install -q 'numpy<2.0'")

import numpy as _np
if int(_np.__version__.split('.')[0]) >= 2:
    print(f"\n⚠ NumPy {_np.__version__} is still in memory (installed version is now <2.0).")
    print("  Restarting kernel so NumPy 1.x loads from scratch...")
    print("  ➜ After the restart, re-run THIS cell (Cell 2) again —")
    print("    all packages are already installed so it will finish in ~10 seconds.")
    import os; os.kill(os.getpid(), 9)   # Colab auto-restarts the kernel

print(f"NumPy {_np.__version__} — OK (no restart needed).")

# Patch inspect.getargspec removed in Python 3.11 — required by chumpy/FLAME
import inspect
if not hasattr(inspect, 'getargspec'):
    inspect.getargspec = inspect.getfullargspec

# ── iopath / fvcore ────────────────────────────────────────────────────────────
if not is_installed('iopath') or not is_installed('fvcore'):
    print("Installing iopath and fvcore...")
    run("pip install -q iopath fvcore")
else:
    print("iopath/fvcore already installed — skipping.")

# ── pytorch3d ─────────────────────────────────────────────────────────────────
if not is_installed('pytorch3d'):
    try:
        urllib.request.urlopen(wheel_url)
        prebuilt_available = True
        print(f"✓ Prebuilt PyTorch3D wheel found — fast install (~30s)")
    except:
        prebuilt_available = False
        print("✗ No prebuilt wheel — compiling PyTorch3D from source (~10-15 min)...")
        print("  (Using MAX_JOBS=4 to speed up C++ compilation)")

    if prebuilt_available:
        run(f'pip install -q pytorch3d -f "{wheel_url}"')
    else:
        run('pip install -q "git+https://github.com/facebookresearch/pytorch3d.git@stable"',
            env={"MAX_JOBS": "4"})
    print("PyTorch3D installed.")
else:
    print("PyTorch3D already installed — skipping.")

# ── SMIRK requirements ────────────────────────────────────────────────────────
smirk_pkgs_missing = any(not is_installed(m) for m in
                         ['mediapipe', 'skimage', 'timm', 'albumentations',
                          'omegaconf', 'chumpy', 'gdown'])
if smirk_pkgs_missing:
    print("Installing SMIRK requirements...")
    run("pip install -q "
        "mediapipe>=0.10.13 "
        "scikit-image==0.22.0 "
        "timm==0.9.16 "
        "albumentations==1.3.0 "
        "omegaconf==2.3.0 "
        "chumpy "
        "gdown")

else:

    print("SMIRK requirements already installed — skipping.")print("\n✓ All dependencies ready.")


In [ ]:
# ── Cell 3: Clone SMIRK repo ──────────────────────────────────────────────────
import os, subprocess

if not os.path.exists('/content/smirk/.git'):
    subprocess.run('git clone https://github.com/georgeretsi/smirk.git /content/smirk',
                   shell=True, check=True)
    print("Cloned SMIRK repo.")
else:
    print("SMIRK repo already cloned.")

os.chdir('/content/smirk')
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Cell 4: Download SMIRK pretrained checkpoint ──────────────────────────────
import os, gdown

os.makedirs('pretrained_models', exist_ok=True)
checkpoint_path = 'pretrained_models/SMIRK_em1.pt'

if not os.path.exists(checkpoint_path):
    print("Downloading SMIRK checkpoint (~140 MB)...")
    gdown.download(id='1T65uEd9dVLHgVw5KiUYL66NUee-MCzoE', output=checkpoint_path, quiet=False)
    print("Checkpoint downloaded.")
else:
    print("Checkpoint already present.")

assert os.path.exists(checkpoint_path), "Checkpoint download failed!"

In [ ]:
# ── Cell 5: Upload FLAME 2020 model ───────────────────────────────────────────
#
# FLAME requires a free registration at https://flame.is.tue.mpg.de
# After registering, download  FLAME2020.zip  and upload it here.
#
import os, zipfile
from google.colab import files

FLAME_PKL = 'assets/FLAME2020/generic_model.pkl'

if not os.path.exists(FLAME_PKL):
    print("Please upload FLAME2020.zip when the file picker appears...")
    uploaded = files.upload()  # opens file picker

    zip_name = next(iter(uploaded))   # filename of the uploaded file
    zip_path = f'/content/smirk/{zip_name}'

    # Write uploaded bytes to disk (Colab puts them in memory)
    with open(zip_path, 'wb') as f:
        f.write(uploaded[zip_name])

    print(f"Extracting {zip_name}...")
    os.makedirs('assets', exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('assets/')   # zip contains FLAME2020/ subfolder
    os.remove(zip_path)
    print("FLAME extracted.")
else:
    print("FLAME model already present.")

# Also download the mediapipe face landmarker
mp_task = 'assets/face_landmarker.task'
if not os.path.exists(mp_task):
    print("Downloading MediaPipe face landmarker...")
    os.system(
        "wget -q https://storage.googleapis.com/mediapipe-models/"
        "face_landmarker/face_landmarker/float16/latest/face_landmarker.task "
        f"-O {mp_task}"
    )
    print("MediaPipe landmarker downloaded.")

assert os.path.exists(FLAME_PKL), f"FLAME model not found at {FLAME_PKL}"
print("\nAll model files ready.")

In [ ]:

# ── Cell 6: Load SMIRK model ──────────────────────────────────────────────────
import torch, sys, os, subprocess, importlib
import numpy as _np

# Verify numpy version — must be <2.0 for FLAME pickle to load correctly
if int(_np.__version__.split('.')[0]) >= 2:
    raise RuntimeError(
        f"NumPy {_np.__version__} detected! FLAME requires NumPy <2.0.\n"
        "Please re-run Cell 2 — it will install numpy<2.0 and restart the kernel."
    )
print(f"NumPy {_np.__version__} — OK.")

SMIRK_DIR = '/content/smirk'

# Re-clone if src/ is missing (handles incomplete clones and runtime resets)
if not os.path.exists(os.path.join(SMIRK_DIR, 'src')):
    print("SMIRK src/ not found — cloning repo...")
    subprocess.run(
        f'rm -rf {SMIRK_DIR} && git clone --depth 1 https://github.com/georgeretsi/smirk {SMIRK_DIR}',
        shell=True, check=True
    )
    print("Cloned.")

os.chdir(SMIRK_DIR)

# Remove any stale sys.path entries and re-insert at front
sys.path = [p for p in sys.path if p != SMIRK_DIR]
sys.path.insert(0, SMIRK_DIR)

# Purge any stale cached 'src' imports from sys.modules
for _key in list(sys.modules.keys()):
    if _key == 'src' or _key.startswith('src.'):
        del sys.modules[_key]

# Create __init__.py files so local src/ is a proper package, not a namespace pkg
for _pkg_dir in [
    os.path.join(SMIRK_DIR, 'src'),
    os.path.join(SMIRK_DIR, 'src', 'FLAME'),
    os.path.join(SMIRK_DIR, 'src', 'renderer'),
    os.path.join(SMIRK_DIR, 'src', 'models'),
    os.path.join(SMIRK_DIR, 'src', 'utils'),
]:
    _init = os.path.join(_pkg_dir, '__init__.py')
    if os.path.isdir(_pkg_dir) and not os.path.exists(_init):
        open(_init, 'w').close()
        print(f"Created {_init}")

# Invalidate importer cache so Python re-scans dirs after __init__.py creation
importlib.invalidate_caches()

# Patch inspect.getargspec removed in Python 3.11 — required by chumpy/FLAME
import inspect
if not hasattr(inspect, 'getargspec'):
    inspect.getargspec = inspect.getfullargspec

# Sanity-check before importing
assert os.path.isdir(os.path.join(SMIRK_DIR, 'src')), f"src/ still missing in {SMIRK_DIR}"
assert os.path.exists(os.path.join(SMIRK_DIR, 'src', '__init__.py')), "src/__init__.py missing!"
print(f"sys.path[0] = {sys.path[0]}")
print(f"src/ contents: {os.listdir(os.path.join(SMIRK_DIR, 'src'))[:8]}")

from src.smirk_encoder import SmirkEncoder
from src.FLAME.FLAME import FLAME
from src.renderer.renderer import Renderer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CHECKPOINT = 'pretrained_models/SMIRK_em1.pt'

# Load encoder
smirk_encoder = SmirkEncoder().to(DEVICE)
checkpoint = torch.load(CHECKPOINT, map_location=DEVICE)
encoder_state = {k.replace('smirk_encoder.', ''): v
                 for k, v in checkpoint.items() if 'smirk_encoder' in k}
smirk_encoder.load_state_dict(encoder_state)
smirk_encoder.eval()

# Load FLAME and renderer
flame    = FLAME().to(DEVICE)
renderer = Renderer().to(DEVICE)
print(f"Model loaded on {DEVICE}")


In [ ]:
    # ── Cell 7: Inference helper function ─────────────────────────────────────────
    import cv2, numpy as np, torch
    import torch.nn.functional as F
    from skimage.transform import estimate_transform, warp
    from utils.mediapipe_utils import run_mediapipe


    def crop_face(frame, landmarks, scale=1.4, image_size=224):
        left   = np.min(landmarks[:, 0])
        right  = np.max(landmarks[:, 0])
        top    = np.min(landmarks[:, 1])
        bottom = np.max(landmarks[:, 1])
        old_size = (right - left + bottom - top) / 2
        center   = np.array([right - (right - left) / 2.0,
                            bottom - (bottom - top) / 2.0])
        size = int(old_size * scale)
        src_pts = np.array([[center[0] - size/2, center[1] - size/2],
                            [center[0] - size/2, center[1] + size/2],
                            [center[0] + size/2, center[1] - size/2]])
        DST_PTS = np.array([[0, 0], [0, image_size - 1], [image_size - 1, 0]])
        return estimate_transform('similarity', src_pts, DST_PTS)


    @torch.no_grad()
    def run_smirk(image_bgr, crop=True):
        """
        Run SMIRK inference on a BGR image (as loaded by cv2).
        Returns a side-by-side numpy RGB image: [input | 3D overlay].
        """
        image_size = 224
        orig_h, orig_w, _ = image_bgr.shape

        kpt = run_mediapipe(image_bgr)
        if kpt is None:
            raise ValueError("MediaPipe could not detect a face in the image.")

        kpt2d = kpt[..., :2]

        if crop:
            tform = crop_face(image_bgr, kpt2d, scale=1.4, image_size=image_size)
            cropped = warp(image_bgr, tform.inverse,
                            output_shape=(image_size, image_size),
                            preserve_range=True).astype(np.uint8)
            kpt2d_c = (tform.params @
                        np.hstack([kpt2d, np.ones((kpt2d.shape[0], 1))]).T).T[:, :2]
        else:
            cropped = cv2.resize(image_bgr, (image_size, image_size))
            kpt2d_c = kpt2d
            tform   = None

        inp = (torch.tensor(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
            .permute(2, 0, 1).unsqueeze(0).float() / 255.0).to(DEVICE)

        outputs = smirk_encoder(inp)
        flame_out   = flame.forward(outputs)
        render_out  = renderer.forward(
            flame_out['vertices'], outputs['cam'],
            landmarks_fan=flame_out['landmarks_fan'],
            landmarks_mp=flame_out['landmarks_mp']
        )
        rendered = render_out['rendered_img']  # (1,3,224,224)

        if crop and tform is not None:
            rend_np = (rendered.squeeze(0).permute(1,2,0)
                            .cpu().numpy() * 255).astype(np.uint8)
            rend_orig = warp(rend_np, tform,
                            output_shape=(orig_h, orig_w),
                            preserve_range=True).astype(np.uint8)
            rend_t = (torch.tensor(rend_orig).permute(2,0,1)
                    .unsqueeze(0).float() / 255.0)
            orig_t = (torch.tensor(
                    cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
                    .permute(2,0,1).unsqueeze(0).float() / 255.0)
            grid = torch.cat([orig_t, rend_t], dim=3)
        else:
            grid = torch.cat([inp.cpu(), rendered.cpu()], dim=3)

        out = (grid.squeeze(0).permute(1,2,0).numpy() * 255).astype(np.uint8)
        return out  # RGB


    print("Inference function ready.")

In [ ]:
# ── Cell 8: Run on a built-in sample image ────────────────────────────────────
import cv2
from IPython.display import display
from PIL import Image
import os

# SMIRK ships sample images in samples/
sample_images = sorted([
    f for f in os.listdir('samples')
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])
print("Available sample images:", sample_images)

# Find the first image where MediaPipe can detect a face
result_rgb = None
INPUT = None
for fname in sample_images:
    candidate = f'samples/{fname}'
    print(f"Trying: {candidate}")
    img = cv2.imread(candidate)
    try:
        result_rgb = run_smirk(img, crop=True)
        INPUT = candidate
        print(f"✓ Face detected in {candidate}")
        break
    except ValueError:
        print(f"  No face detected — skipping.")

if result_rgb is None:
    raise RuntimeError("No face detected in any sample image. Upload your own image in Cell 9.")

pil_img = Image.fromarray(result_rgb)
display(pil_img)
print(f"\nInput: {INPUT}")
print(f"Output size: {pil_img.size}  (original | 3D mesh overlay)")


In [ ]:
# ── Cell 9: Run on your own image ─────────────────────────────────────────────
from google.colab import files
import cv2, numpy as np
from PIL import Image
from IPython.display import display
import io

print("Upload any face image (JPG/PNG):")
uploaded = files.upload()

for fname, data in uploaded.items():
    arr = np.frombuffer(data, dtype=np.uint8)
    image_bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)

    print(f"\nProcessing {fname} ({image_bgr.shape[1]}×{image_bgr.shape[0]})...")
    result_rgb = run_smirk(image_bgr, crop=True)

    pil_result = Image.fromarray(result_rgb)
    display(pil_result)

    # Save output
    out_name = f'smirk_output_{fname}'
    pil_result.save(out_name)
    print(f"Saved: {out_name}")
    files.download(out_name)

In [ ]:
# ── Cell 10 (Optional): Use SMIRK neural image translator ────────────────────
#
# The SMIRK checkpoint also includes a neural image-to-image generator
# that reconstructs a photorealistic face on top of the 3D mesh.
#
import torch
from src.smirk_generator import SmirkGenerator
import src.utils.masking as masking_utils
from datasets.base_dataset import create_mask

smirk_generator = SmirkGenerator(in_channels=6, out_channels=3,
                                  init_features=32, res_blocks=5).to(DEVICE)
checkpoint = torch.load('pretrained_models/SMIRK_em1.pt', map_location=DEVICE)
gen_state = {k.replace('smirk_generator.', ''): v
             for k, v in checkpoint.items() if 'smirk_generator' in k}
smirk_generator.load_state_dict(gen_state)
smirk_generator.eval()
print("Neural generator loaded.")

In [ ]:
# ── Cell 11 (Optional): Neural generator inference ───────────────────────────
# Run on the same sample, but now output is a photorealistic reconstruction
import cv2, torch, numpy as np, os
from skimage.transform import warp
from PIL import Image
from IPython.display import display
from utils.mediapipe_utils import run_mediapipe
from datasets.base_dataset import create_mask

# Find a sample image where MediaPipe can detect a face.
# Use INPUT from Cell 8 only if it's still valid; otherwise scan samples/.
def _find_face_image():
    candidates = []
    _inp = globals().get('INPUT')
    if _inp:
        candidates.append(_inp)
    for _f in sorted(os.listdir('samples')):
        if _f.lower().endswith(('.png', '.jpg', '.jpeg')):
            _p = f'samples/{_f}'
            if _p not in candidates:
                candidates.append(_p)
    for _p in candidates:
        _img = cv2.imread(_p)
        if _img is not None and run_mediapipe(_img) is not None:
            return _p
    return None

_input_path = _find_face_image()
if _input_path is None:
    raise RuntimeError("No sample image with a detectable face found. Run Cell 8 first.")

print(f"Using: {_input_path}")
image_bgr = cv2.imread(_input_path)

image_size = 224
mask_ratio_mul = 5
mask_ratio = 0.01
mask_dilation_radius = 10

kpt = run_mediapipe(image_bgr)[..., :2]

tform = crop_face(image_bgr, kpt, scale=1.4, image_size=image_size)
cropped = warp(image_bgr, tform.inverse,
                output_shape=(image_size, image_size),
                preserve_range=True).astype(np.uint8)
kpt_c = (tform.params @ np.hstack([kpt, np.ones((kpt.shape[0], 1))]).T).T[:, :2]

inp = (torch.tensor(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
       .permute(2,0,1).unsqueeze(0).float()/255.0).to(DEVICE)

with torch.no_grad():
    outputs    = smirk_encoder(inp)
    flame_out  = flame.forward(outputs)
    render_out = renderer.forward(
        flame_out['vertices'], outputs['cam'],
        landmarks_fan=flame_out['landmarks_fan'],
        landmarks_mp=flame_out['landmarks_mp']
    )
    rendered = render_out['rendered_img']

    hull_mask = torch.from_numpy(
        create_mask(kpt_c, (image_size, image_size))
    ).float().unsqueeze(0).to(DEVICE)

    face_probs = masking_utils.load_probabilities_per_FLAME_triangle()
    rendered_mask = 1 - (rendered == 0).all(dim=1, keepdim=True).float()
    tmask_ratio = mask_ratio * mask_ratio_mul
    npoints, _ = masking_utils.mesh_based_mask_uniform_faces(
        render_out['transformed_vertices'],
        flame_faces=flame.faces_tensor,
        face_probabilities=face_probs,
        mask_ratio=tmask_ratio
    )
    pmask = torch.zeros_like(rendered_mask)
    rsing  = torch.randint(0, 2, (npoints.size(0),)).to(DEVICE) * 2 - 1
    rscale = torch.rand((npoints.size(0),)).to(DEVICE) * (mask_ratio_mul - 1) + 1
    rbound = (npoints.size(1) * (1/mask_ratio_mul) * (rscale ** rsing)).long()
    for bi in range(npoints.size(0)):
        pmask[bi, :, npoints[bi, :rbound[bi], 1], npoints[bi, :rbound[bi], 0]] = 1

    extra_points = inp * pmask
    masked_img   = masking_utils.masking(inp, hull_mask, extra_points,
                                          mask_dilation_radius,
                                          rendered_mask=rendered_mask)
    gen_input       = torch.cat([rendered, masked_img], dim=1)
    reconstructed   = smirk_generator(gen_input)

    grid = torch.cat([inp.cpu(), rendered.cpu(), reconstructed.cpu()], dim=3)

out = (grid.squeeze(0).permute(1,2,0).numpy() * 255).astype(np.uint8)
display(Image.fromarray(out))
print(f"Input: {_input_path}")
print("Left: input  |  Middle: 3D mesh  |  Right: neural reconstruction")
